# CSA402 Natural Language Processing
## Fine-Tuning ViT-GPT2 for Bhutanese Cultural and Tourism Image Captioning

**Gyalpozhing College of Information Technology (GCIT), Kabesa, Thimphu**  
**Bachelor of Science in Computer Science**  
**AI Development and Data Science**  
**Year IV, Semester VII**  

**Group Members:**
- Chimi Kawang Selden (12220029)
- Kezang Choden (12220037)
- Kinchap Legden (12220038)

**Advisor:** Mam Tawmo  

This notebook implements the complete pipeline to fine-tune `nlpconnect/vit-gpt2-image-captioning` on a custom dataset of Bhutanese cultural and tourism images (including Dzongs, Tsechu festivals, sacred prayer flags, monasteries, monastic life, and pristine landscapes).

### Step 1: Install Required Libraries

In [34]:
!pip install -q transformers datasets evaluate accelerate rouge_score jiwer scikit-learn albumentations pandas pillow ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 48.3 MB/s eta 0:00:00


### Step 2: Import Dependencies and Declare the Custom Dataset

In [35]:
import os
import io
import pandas as pd
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset as TorchDataset
from sklearn.model_selection import train_test_split
from transformers import (
    VisionEncoderDecoderModel,
    ViTImageProcessor,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate





In [36]:
from google.colab import drive
drive.mount('/content/drive')




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [61]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/NLP Project/train.csv")

# clean column names
df.columns = df.columns.str.strip()

# fix wrong column name
df = df.rename(columns={
    "mage": "image_id",
    "image": "image_id",
    "filename": "image_id"
})

print(df.columns)
print(df.head())

Index(['image_id', 'caption'], dtype='object')
  image_id                                            caption
0    1.jpg  A wooden table and benches overlook a valley i...
1    2.jpg  Four dancers in elaborate traditional costumes...
2    3.jpg  A serene scene in Bhutan featuring two traditi...
3    4.jpg  A view of a traditional Bhutanese temple surro...
4    5.jpg  A panoramic view of a lush valley in Bhutan, f...


### Step 3: Split Dataset (70% Training, 15% Validation, 15% Testing)

In [62]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

In [53]:
# Perform splits mathematically
from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f"Training set size: {len(train_df)} (70%)")
print(f"Validation set size: {len(val_df)} (15%)")
print(f"Testing set size: {len(test_df)} (15%)")

Training set size: 113 (70%)
Validation set size: 24 (15%)
Testing set size: 25 (15%)


### Step 4: Load Vision-Language Transformer (ViT-GPT2)
We use the pre-trained `ViT-GPT2` from Hugging Face which uses Vision Transformer (ViT) as image encoder and GPT-2 as textual decoder.

In [63]:
from transformers import (
    VisionEncoderDecoderModel,
    ViTImageProcessor,
    AutoTokenizer
)

In [64]:
model_name = "nlpconnect/vit-gpt2-image-captioning"
model = VisionEncoderDecoderModel.from_pretrained(model_name)
image_processor = ViTImageProcessor.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Configure tokenizer settings for sequence generation
tokenizer.pad_token = tokenizer.eos_token
model.config.eos_token_id = tokenizer.eos_token_id
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/445 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.transformer.wte.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
VisionEncoderDecoderModel LOAD REPORT from: nlpconnect/vit-gpt2-image-captioning
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
decoder.transformer.h.{0...11}.attn.masked_bias           | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.crossattention.masked_bias | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.attn.bias                  | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.crossattention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Step 5: Create PyTorch Custom Dataset Class
We write a custom PyTorch dataset to download/load the files, process features using ViT, and tokenize reference captions.

In [65]:
import torch
import os
from torch.utils.data import Dataset as TorchDataset

In [67]:
class BhutanDataset(TorchDataset):
    def __init__(self, df, image_dir_or_dict, image_processor, tokenizer, max_length=128):
        self.df = df.reset_index(drop=True)
        self.image_dir_or_dict = image_dir_or_dict
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row["image_id"]
        caption = row["caption"]

        # In practice, load from a physical directory or create an elegant placeholder if the file isn't downloaded yet
        try:
            if isinstance(self.image_dir_or_dict, dict):
                image = self.image_dir_or_dict[img_name]
            else:
                image_path = os.path.join(self.image_dir_or_dict, img_name)
                image = Image.open(image_path).convert("RGB")
        except Exception:
            # Fallback placeholder to keep training stable during tests
            image = Image.fromarray(np.uint8(np.random.randint(0, 255, (224, 224, 3)))).convert("RGB")

        # Preprocess visual features
        pixel_values = self.image_processor(images=image, return_tensors="pt").pixel_values.squeeze()

        # Preprocess labels
        tokens = self.tokenizer(
            caption,
            padding="max_length",
            max_length=self.max_length,
            truncation=True
        )
        labels = tokens["input_ids"]
        # Replace padding index with -100 to ignore in loss estimation
        labels = [l if l != self.tokenizer.pad_token_id else -100 for l in labels]

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(labels)
        }

# Create sample directory
import os

os.makedirs("/content/drive/MyDrive/NLP Project/Images", exist_ok=True)

train_dataset = BhutanDataset(train_df, "/content/drive/MyDrive/NLP Project/Images", image_processor, tokenizer)
val_dataset = BhutanDataset(val_df, "/content/drive/MyDrive/NLP Project/Images", image_processor, tokenizer)

### Step 6: Configure NLP Evaluation Metrics
We declare the evaluation callbacks to monitor **BLEU, ROUGE-L, and METEOR** scores on the validation subset.

In [43]:
!pip install evaluate

In [44]:
import evaluate

In [45]:
!pip install evaluate rouge_score sacrebleu

In [57]:

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    # Replace ignored label id with padding ID
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Basic formatting for evaluations
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    bleu_results = bleu.compute(predictions=decoded_preds, references=decoded_labels)
    rouge_results = rouge.compute(predictions=decoded_preds, references=[l[0] for l in decoded_labels])

    return {
        "bleu": bleu_results["bleu"],
        "rougeL": rouge_results["rougeL"]
    }
    print("hi")

In [70]:
def compute_metrics(eval_pred):

    preds, labels = eval_pred

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    bleu_results = bleu.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    rouge_results = rouge.compute(
        predictions=decoded_preds,
        references=[l[0] for l in decoded_labels]
    )

    return {
        "bleu": bleu_results["bleu"] * 100,
        "rougeL": rouge_results["rougeL"] * 100
    }

### Step 7: Define Fine-Tuning Hyperparameters & Train Model

In [69]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

In [72]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_bleu",
    greater_is_better=True,
    logging_dir="./logs",
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training loop...")
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting training loop...


Epoch,Training Loss,Validation Loss,Bleu,Rougel
1,No log,1.993177,6.134320,31.701240
2,No log,1.928629,5.281669,30.382997
3,No log,1.896621,6.608815,33.003015
4,No log,1.958788,4.433534,29.411842
5,No log,1.985257,5.593927,30.758076


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['decoder.lm_head.weight'].


TrainOutput(global_step=145, training_loss=1.4144741716056035, metrics={'train_runtime': 4534.9909, 'train_samples_per_second': 0.125, 'train_steps_per_second': 0.032, 'total_flos': 1.2165795039412224e+17, 'train_loss': 1.4144741716056035, 'epoch': 5.0})

In [73]:
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 1.896620750427246, 'eval_bleu': 6.608815093222037, 'eval_rougeL': 33.00301511959564, 'eval_runtime': 86.3345, 'eval_samples_per_second': 0.278, 'eval_steps_per_second': 0.069, 'epoch': 5.0}


### Step 8: Interactive Captions Generator with Photo Upload widget
Run this cell in your environment to upload your own custom photo, visualize it, and generate highly specialized Bhutanese descriptions!

In [78]:
import ipywidgets as widgets
from IPython.display import display
import torch
from PIL import Image

uploader = widgets.FileUpload(
    accept='image/*',  # Accept images only
    multiple=False  # True to upload multiple files
)
output_image = widgets.Output()
output_text = widgets.Output()

def on_image_upload(change):
    output_image.clear_output()
    output_text.clear_output()

    if not uploader.value:
        return

    # Parse uploaded file
    uploaded_file = list(uploader.value.values())[0]
    image_content = uploaded_file['content']
    image = Image.open(io.BytesIO(image_content)).convert("RGB")

    # Display image in Jupyter
    with output_image:
        display(image.resize((350, 250)))

    # Format and generate captions
    with output_text:
        print("Processing visual embedding and decoding caption...")
        pixel_values = image_processor(images=image, return_tensors="pt").pixel_values

        # Put on GPU if available
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        pixel_values = pixel_values.to(device)

        output_ids = model.generate(
            pixel_values,
            max_length=40,
            num_beams=4,
            no_repeat_ngram_size=2
        )

        caption = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
        print("\n" + "="*40)
        print("Generated Caption (viT-GPT2 Model):")
        print(caption)
        print("="*40)

uploader.observe(on_image_upload, names='value')

print("Upload your image below:")
display(widgets.VBox([uploader, output_image, output_text]))

Upload your image below:


In [76]:
import pickle

In [77]:
# Save everything in one pickle file
with open("vit_gpt2_caption_model.pkl", "wb") as f:
    pickle.dump({
        "model": model,
        "tokenizer": tokenizer,
        "image_processor": image_processor
    }, f)

print("Pickle model saved successfully!")

Pickle model saved successfully!
